# Spectra of J_par and J_perp across the beta scan

One notebook for what used to be three (`JparSpectrum`, `JperpSpectrum`, `JperpSpectrum40`, which differed only in the
component and the snapshot; they are in `archive/jpar_jperp_spectra/`). Choose `COMPONENT` and `TIME` in the settings
cell. The figure is saved as `J<component>_Spectrum_beta_scan_t<TIME>.png` in `results/jpar_jperp/JparJperpSpectrum/`,
the same names the three notebooks used.

* J_par = J . B / |B|; its spectrum is computed directly.
* J_perp: spectrum of |J|^2 (sum of the Jx, Jy, Jz spectra) minus the J_par spectrum, as in the old notebooks.

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt

# shared code and paths (see README.md and shared/)
import sys
from pathlib import Path
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "shared" / "paths.py").exists())
sys.path.insert(0, str(REPO))
from shared.fields import compute_radial_spectrum, partial_x_fft, partial_y_fft
from shared.paths import DATA_ROOT, results_dir
RESULTS = results_dir("jpar_jperp", "JparJperpSpectrum")

In [ ]:
COMPONENT = "par"            # "par" for J_par, "perp" for J_perp
TIME      = 200              # snapshot (the old notebooks: J_par at t = 200, J_perp at t = 200 and t = 40)
betas     = [0.0625, 1, 16]
dx = dy   = 0.0625

In [ ]:
spectra = {}

for beta in betas:

    base = f"{DATA_ROOT}/beta_p_{beta}/beta_e_1"

    File_Bx = f"{base}/Bx_ApJ_t{TIME}.h5"
    File_By = f"{base}/By_ApJ_t{TIME}.h5"
    File_Bz = f"{base}/Bz_ApJ_t{TIME}.h5"

    with h5py.File(File_Bx, 'r') as fBx, \
         h5py.File(File_By, 'r') as fBy, \
         h5py.File(File_Bz, 'r') as fBz:

        data_Bx = fBx['DS1'][:].T
        data_By = fBy['DS1'][:].T
        data_Bz = fBz['DS1'][:].T

    # Current components
    data_Jx = partial_y_fft(data_Bz, dx)
    data_Jy = -partial_x_fft(data_Bz, dy)
    data_Jz = partial_x_fft(data_By, dy) - partial_y_fft(data_Bx, dx)

    Jpar = (data_Jx*data_Bx + data_Jy*data_By + data_Jz*data_Bz) / \
           np.sqrt(data_Bx**2 + data_By**2 + data_Bz**2)

    # Spectra
    spectrum_Jpar = compute_radial_spectrum(Jpar, dx, dy)

    if COMPONENT == "par":
        spectra[beta] = spectrum_Jpar
    else:
        spectrum_Jx = compute_radial_spectrum(data_Jx, dx, dy)
        spectrum_Jy = compute_radial_spectrum(data_Jy, dx, dy)
        spectrum_Jz = compute_radial_spectrum(data_Jz, dx, dy)

        spectrum_J = np.copy(spectrum_Jz)
        spectrum_J['sp'] = spectrum_Jx['sp'] + spectrum_Jy['sp'] + spectrum_Jz['sp']

        spectrum_Jperp = np.copy(spectrum_Jz)
        spectrum_Jperp['sp'] = spectrum_J['sp'] - spectrum_Jpar['sp']

        spectra[beta] = spectrum_Jperp

In [ ]:
plt.figure()

for beta in betas:
    spec = spectra[beta]

    plt.loglog(
        spec['kk'],
        spec['sp'],
        label=fr"$\beta_p = {beta}$"
    )

plt.xlabel(r"$k$")
plt.ylabel(r"$P(k)$")
plt.xlim(5e-2, 30)
plt.ylim(1e-6, 1)

plt.legend()
plt.tight_layout()

plt.savefig(RESULTS / f"J{COMPONENT}_Spectrum_beta_scan_t{TIME}.png", dpi=300)
plt.show()